## Extract Junior Authors from Matched Awards

In [10]:
import pandas as pd
import requests
import json
import time
import ast

matched_df = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\huang_matched_openalex.csv')
matched_df = matched_df[matched_df['openalex_id'].notna()].copy()
matched_df = matched_df.reset_index(drop=True)
if 'award_id' not in matched_df.columns:
    matched_df['award_id'] = matched_df.index + 1

print(f"Working with {len(matched_df)} matched awards across {matched_df['conference'].nunique()} conferences")
print(matched_df['conference'].value_counts())


Working with 903 matched awards across 31 conferences
conference
CHI           185
ICSE           88
FSE            63
UIST           34
PLDI           32
STOC           31
FOCS           30
ACL            26
AAAI           26
IJCAI          25
INFOCOM        24
SOSP           22
VLDB           21
KDD            20
CVPR           20
PODS           20
SIGIR          19
WWW            19
SIGMOD         19
JCDL           19
ICML           19
OSDI           19
CIKM           16
NeurIPS        15
ICWSM          15
SIGMETRICS     15
SIGCOMM        14
SODA           14
ICCV           12
S&P            11
MOBICOM        10
Name: count, dtype: int64


In [11]:
def get_author_works(author_id):
    """Retrieve all works for a given author with pagination."""
    all_works = []
    cursor = '*'
    while cursor:
        url = 'https://api.openalex.org/works'
        params = {
            'filter': f'author.id:{author_id}',
            'per-page': 200,
            'cursor': cursor,
            'mailto': 'shaheryar.4822@student.uu.se'
        }
        response = requests.get(url, params=params)
        data = response.json()
        results = data.get('results', [])
        all_works.extend(results)
        cursor = data.get('meta', {}).get('next_cursor')
        if cursor:
            time.sleep(0.05)
    return all_works

def calculate_career_age(works, reference_year):
    """Calculate years since first publication."""
    years = [w['publication_year'] for w in works if w.get('publication_year')]
    if not years:
        return None
    return reference_year - min(years)

def parse_authorships(raw):
    """Handle both JSON string and list formats."""
    if isinstance(raw, list):
        return raw
    if not isinstance(raw, str) or raw.strip() in ('', 'nan', '[]'):
        return []
    try:
        return json.loads(raw)
    except:
        try:
            return ast.literal_eval(raw)
        except:
            return []


In [12]:
junior_authors = []
processed_authors = {}

for idx, row in matched_df.iterrows():
    award_year = int(row['year'])
    conference = row['conference']
    title = str(row.get('title', ''))[:50]
    authorships = parse_authorships(row.get('authorships', '[]'))

    print(f"\n{idx+1}/{len(matched_df)} {conference} {award_year} {title}...")

    for pos, auth in enumerate(authorships[:3], start=1):
        author_id = auth.get('author', {}).get('id', '')
        author_name = auth.get('author', {}).get('display_name', 'Unknown')

        if not author_id:
            print(f"  ⚠ No author ID at position {pos}")
            continue

        short_id = author_id.split('/')[-1]

        if short_id in processed_authors:
            career_age = processed_authors[short_id].get('career_age')
            if career_age is not None and career_age <= 5:
                print(f"  {author_name} already processed")
            else:
                print(f"  {author_name} already processed")
            continue

        works = get_author_works(short_id)
        career_age = calculate_career_age(works, award_year)
        pub_count = len(works)

        processed_authors[short_id] = {
            'career_age': career_age,
            'pub_count': pub_count
        }

        if career_age is None:
            print(f"  {author_name} pos {pos}... No publication year data")
            continue

        is_junior = career_age <= 5
        status = f"✓ JUNIOR age {career_age}y, {pub_count} pubs" if is_junior else f"Senior age {career_age}y"
        print(f"  {author_name} pos {pos}... {status}")

        if is_junior:
            institutions = [i.get('display_name', '') for i in auth.get('institutions', [])]
            countries    = [i.get('country_code', '') for i in auth.get('institutions', [])]
            junior_authors.append({
                'award_id':            row.get('award_id', idx + 1),
                'conference':          conference,
                'award_year':          award_year,
                'paper_title':         row.get('title', ''),
                'openalex_id':         row.get('openalex_id', ''),
                'author_id':           author_id,
                'author_name':         author_name,
                'author_position':     pos,
                'is_corresponding':    auth.get('is_corresponding', False),
                'career_age':          career_age,
                'pub_count_at_award':  pub_count,
                'institutions':        '; '.join(institutions),
                'countries':           '; '.join(countries),
                'match_route':         row.get('match_route', '')
            })

    if (idx + 1) % 50 == 0:
        print(f"\n── Progress: {len(junior_authors)} juniors found from {idx+1} awards ──\n")

print(f"\n══ DONE: {len(junior_authors)} junior authors found across {len(matched_df)} awards ══")



1/903 AAAI 2018 ...
  Chenjun Xiao pos 1... Senior age 6y
  Jincheng Mei pos 2... ✓ JUNIOR age 4y, 29 pubs
  Martin Müller pos 3... Senior age 52y

2/903 ACL 2018 ...
  John Hale pos 1... Senior age 88y
  Chris Dyer pos 2... Senior age 48y
  Adhiguna Kuncoro pos 3... ✓ JUNIOR age 2y, 38 pubs

3/903 ACL 2018 ...
  Sudha Rao pos 1... Senior age 7y
  Hal Daumé pos 2... Senior age 17y

4/903 ACL 2018 ...
  Alexandre Cremers pos 1... Senior age 7y

5/903 CHI 2018 ...
  Yongkwan Kim pos 1... Senior age 13y
  Sang-Gyun An pos 2... ✓ JUNIOR age 1y, 8 pubs
  Joon Hyub Lee pos 3... Senior age 6y

6/903 CHI 2018 ...
  Mikko Kytö pos 1... Senior age 21y
  Barrett Ens pos 2... Senior age 10y
  Thammathip Piumsomboon pos 3... Senior age 7y

7/903 CHI 2018 ...
  Zhicheng Liu pos 1... Senior age 15y
  J. Thompson pos 2... Senior age 58y
  Alan M. Wilson pos 3... Senior age 94y

8/903 CHI 2018 ...
  Jayson Althofer pos 1... Senior age 19y
  Brian Musgrove pos 2... Senior age 32y

9/903 CHI 2018 ...
 

In [14]:
junior_df = pd.DataFrame(junior_authors)
print(f"Total junior authors: {len(junior_df)}")
print(f"Unique junior authors: {junior_df['author_id'].nunique()}")
print("\nBy conference:")
print(junior_df['conference'].value_counts())
print("\nCareer age distribution:")
print(junior_df['career_age'].value_counts().sort_index())
print("\nMatch routes:")
print(junior_df['match_route'].value_counts())

output_path = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_matched.csv'
junior_df.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")


Total junior authors: 602
Unique junior authors: 602

By conference:
conference
CHI           118
ICSE           82
FSE            47
UIST           27
PLDI           26
OSDI           22
SOSP           22
ACL            19
VLDB           17
AAAI           16
WWW            16
INFOCOM        16
SIGCOMM        13
ICML           13
KDD            12
SIGMOD         12
CIKM           12
PODS           12
SIGMETRICS     11
CVPR           11
FOCS           11
STOC           10
IJCAI          10
SIGIR           9
S&P             9
MOBICOM         9
NeurIPS         8
ICCV            8
SODA            4
Name: count, dtype: int64

Career age distribution:
career_age
-16      1
-13      1
-2       1
-1       1
 0     130
 1      79
 2      92
 3      86
 4     117
 5      94
Name: count, dtype: int64

Match routes:
match_route
SS→DOI→OA      421
SS→title→OA    146
GS→title→OA     35
Name: count, dtype: int64

Saved to B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched\junior_authors_mat